# 04 · Explaining the model and studying its mistakes: a guided tour of Phase 5

Phases 3 and 4 built a model that ranks pairs, names a behavior and points at five hands. An investigator cannot act
on a bare number, though. They need to know **why** a pair scored high, **which actions** in a hand look planted, and
**what innocent story** they must rule out before accusing anyone. This notebook builds those three explanations
and then turns them on the model itself: where does it go wrong, and what do the mistakes have in common?

Same structure as notebooks 01–03: 🎯 Question · 🤔 Why it matters · 🔬 Method · 📊 What we saw · 💡 Inference ·
➡️ Consequence.

### Sections
1. Three kinds of explanation, and the one property that makes SHAP trustworthy
2. What drives the risk score overall
3. One pair, explained
4. Evidence reason tags: do they point at the planted actions?
5. The break-even cut: counting mistakes honestly
6. False positives and the decoys: do innocent stories explain them?
7. False negatives: what the missed colluders have in common
8. Behavior and evidence mistakes by family
9. What goes on the public model card
10. Summary and self-check

> Outputs use competition data, so they stay local (the committed notebook has no outputs). Run
> `python -m suspoker.train` and `python -m suspoker.explain` first.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from suspoker import error_analysis as E
from suspoker.config import ARTIFACTS_DIR, RAW_DIR
from suspoker.explain import (
    BENIGN,
    EXPLAIN_DIR,
    TAG_ORDER,
    TAGS,
    benign_alternatives,
    contributions,
    family_importance,
    feature_info,
    load_reference,
    tag_flags,
)
from suspoker.modeling import FAMILIES, KEYS, hand_context, keyed, matrix
from suspoker.pipeline import FEATURES_DIR
from suspoker.predict import Models, evaluation_pairs

pl.Config(tbl_rows=30, tbl_width_chars=180, fmt_str_lengths=80, float_precision=3)
models = Models.load()
feats = models.meta["pair_features"]
oof = E.load_oof()
flagged = E.flag_top_k(E.with_window_features(oof))
print(f"mirror pairs: {oof.height:,} | colluders: {oof['label'].sum()} | pair features: {len(feats)}")

## 1 · Three kinds of explanation, and the one property that makes SHAP trustworthy

| Question an investigator asks | Explanation | Source |
|---|---|---|
| Why is this pair's risk high? Why this behavior? | top features pushing the score **up** and **down** | SHAP on M1 / M2 |
| What happened in this evidence hand? | 1–3 reason tags ("folded the better hand to the partner's bet") | the hand features M4 used |
| Could this be innocent? | the strongest benign-alternative template, with its numbers | percentiles of control features |

🤔 **Why SHAP rather than "feature importance"?** Importance says what the model uses *on average*. An investigator
looks at *one* pair. SHAP gives each feature a contribution for that pair, and the contributions satisfy
**additivity**: base value + Σ contributions = the model's raw score (log-odds). Nothing is left unexplained and
nothing is double-counted. Without additivity, an explanation can sound plausible while having nothing to do with
the number on the screen.

🔬 **Method.** LightGBM ships the exact TreeSHAP algorithm (`pred_contrib=True`, the same algorithm as
`shap.TreeExplainer`, checked in `tests/test_explain.py`). We average contributions over the three seed models.
The cell below checks additivity on 2,000 evaluation pairs.

In [ ]:
pairs = evaluation_pairs(models)
X = matrix(pairs.head(2000), feats)
c = contributions(models.m1, X)
raw = np.mean([b.predict(X, raw_score=True) for b in models.m1], axis=0)
print(f"base value (average raw score): {c[0, -1]:.3f} log-odds")
print(f"largest |base + sum(contributions) - raw score|: {np.abs(c.sum(axis=1) - raw).max():.2e}")

📊 The gap is at floating-point level (~1e-14).

💡 **Inference:** every risk score on the dashboard can be broken down exactly into feature contributions. The
"reasons" are not a separate model guessing at the main one; they are the main model, taken apart.

➡️ **Consequence:** MA5 ("SHAP values for every pair shown") costs one extra `predict` call per pair. `python -m
suspoker.explain` produces them for all 112,540 evaluation pairs in about a minute.

## 2 · What drives the risk score overall

🎯 **Question:** which feature families move the risk score the most on the evaluation pairs?

🔬 **Method:** mean |SHAP| per feature (how far it moves a typical pair's score), summed by family.

In [ ]:
imp = pl.read_parquet(EXPLAIN_DIR / "importance_m1.parquet")
fam = family_importance(imp)
print(fam)
print(imp.head(12).select("feature", "family", "label", "mean_abs_shap"))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(fam["family"][::-1], fam["share"][::-1], color="#4a6fa5")
ax.set_xlabel("share of total mean |SHAP|")
ax.set_title("What moves the risk score (evaluation pairs)")
plt.tight_layout()

In [ ]:
# direction check: does each family push the riskiest pairs up or down?
Xa = matrix(pairs, feats)
ca = contributions(models.m1, Xa)[:, :len(feats)]
riskiest = ca.sum(axis=1) >= np.quantile(ca.sum(axis=1), 0.99)
fam_of = np.array([feature_info(f)[0] for f in feats])
rows = []
for name in fam["family"]:
    s = ca[:, fam_of == name].sum(axis=1)[riskiest]
    rows.append({"family": name, "riskiest_1pct_pushed_down": (s < 0).mean(), "mean_push": s.mean()})
print(pl.DataFrame(rows))

📊 **What we saw:** four families carry about three quarters of the movement: F8 presence (~22%), F7 controls
(~20%), F9 hand suspicion (~19%) and F5 isolation (~15%). The single biggest features are `hs_max` and `hs_top3`,
the hand model's view of the pair's most planted-looking hands.

💡 **Inference, and a trap to avoid:** F7 *controls* (each player's general style) rank high, yet style is not
collusion. The direction check shows how they are used. Among the riskiest 1% of pairs, hand suspicion pushes the
score **up** for 98% of them (+5.7 log-odds on average), but the controls push **down** for about a third, and
their average push is small (+0.1). Controls **change what the relationship features mean**: a fold lift of +0.1
is striking for a calling station and unremarkable for someone who folds everything. They act as context and
brakes, not as evidence. High importance means "the model leans on this", not "this is suspicious".

➡️ **Consequence:** the pair view must show reasons **against** as well as **for**. A list of only the upward
pushes would make every flagged pair look like an open-and-shut case.

## 3 · One pair, explained

🎯 **Question:** what does an investigator actually see for a single high-risk pair?

The cell prints the explanation of the highest-risk evaluation pair. It stays local: the dashboard shows the same
layout for SimRoom pairs only.

In [ ]:
risk_reasons = pl.read_parquet(EXPLAIN_DIR / "risk_reasons.parquet")
beh_reasons = pl.read_parquet(EXPLAIN_DIR / "behavior_reasons.parquet")
pred = pl.read_parquet(ARTIFACTS_DIR / "predictions.parquet")
top = pred.sort("risk", descending=True).row(0, named=True)
print(f"risk {top['risk']:.3f} | predicted behavior {top['predicted_behavior']}")
for d in ("for", "against"):
    print(f"\n-- pushing risk {'up' if d == 'for' else 'down'}")
    for r in risk_reasons.filter((pl.col("id") == top["pair_id"]) & (pl.col("direction") == d)).iter_rows(named=True):
        print(f"  {r['shap']:+.2f}  {r['label']}  (value {r['value']:.3g})")
print("\n-- why this behavior")
for r in beh_reasons.filter((pl.col("id") == top["pair_id"]) & (pl.col("direction") == "for")).iter_rows(named=True):
    print(f"  {r['shap']:+.2f}  {r['label']}")
row = pairs.filter(pl.col("pair_id") == top["pair_id"]).row(0, named=True)
alts = benign_alternatives(row, load_reference())
print("\n-- innocent explanation to rule out:", alts[0]["text"] if alts else "no common innocent pattern fits")

💡 **Reading it like an investigator:** the upward reasons say *what pattern* the model saw; the downward reasons
say *what argues against it*; the benign template names the most likely innocent story and the number behind it.
Together they are a case file to check, not a verdict. The competition's own verification asks for exactly this:
observable behavior plus a plausible benign alternative.

## 4 · Evidence reason tags: do they point at the planted actions?

🎯 **Question:** M4 picks five hands per pair. Its tags say *why* in plain words ("gave up 3.2 bb of expected value
against the partner"). Do the tags actually separate planted hands from ordinary ones, or would they fire on any
hand?

🔬 **Method:** take every candidate hand of the labeled colluding pairs (development period), mark the planted ones
from `development_evidence.csv`, and compare how often each tag fires on planted vs other hands. The ratio is the
tag's **lift**: how much more likely a hand carrying the tag is to be planted.

In [ ]:
labels = keyed(pl.read_csv(RAW_DIR / "development_labels.csv")).filter(pl.col("label") == 1)
planted = pl.read_csv(RAW_DIR / "development_evidence.csv").select("pair_id", "hand_id", pl.lit(1).alias("planted"))
hf = pl.read_parquet(FEATURES_DIR / "hand_features.parquet").filter(pl.col("phase") == 0)
cand = hand_context(hf.join(labels.select("pair_id", *KEYS, "behavior_family"), on=KEYS).sort("pair_id", "hand_idx"))
cand = tag_flags(cand.join(planted, on=["pair_id", "hand_id"], how="left").with_columns(pl.col("planted").fill_null(0)))
rows = []
for fam_name in FAMILIES:
    g = cand.filter(pl.col("behavior_family") == fam_name)
    for k in TAGS:
        p_rate = g.filter(pl.col("planted") == 1)[f"tag_{k}"].mean()
        o_rate = g.filter(pl.col("planted") == 0)[f"tag_{k}"].mean()
        rows.append({"family": fam_name, "tag": k, "planted_rate": p_rate, "other_rate": o_rate,
                     "lift": p_rate / o_rate if o_rate else None})
tag_lift = pl.DataFrame(rows).filter(pl.col("tag") != "contested")
print(f"candidate hands: {cand.height:,}, planted: {cand['planted'].sum():,}")
print(tag_lift.sort("family", "lift", descending=[False, True]).group_by("family", maintain_order=True).head(6))
for fam_name, order in TAG_ORDER.items():
    print(f"tag order for {fam_name}: {', '.join(order)}")

📊 **What we saw** (lift = planted rate ÷ other-hand rate):
- **Directed transfer:** `fold_ahead` (folding the better hand to the partner's bet) has lift ≈ 10. It fires on
  54% of planted hands and 5% of the others. Next come `strong_check` (≈ 8) and `junk_call` / `costly_fold` (≈ 3).
  `ev_gift` fires on 89% of planted hands but also on many ordinary ones (lift ≈ 2.6).
- **Soft play:** `fold_ahead` again (≈ 13), then `pre_fold_ahead`, `costly_call` and `strong_check` (≈ 5–6).
- **Isolation:** the best tags are much weaker: `weak_open` ≈ 3.6, `junk_call` ≈ 2.8, `squeeze` ≈ 2.2. The defining
  move, `open_step_aside` (one opens, the other folds behind), has lift ≈ **1.6**. It fires on 47% of planted
  hands, but it is also how ordinary hands go.

💡 **Inference:** a tag with high lift is a faithful summary of what made a hand suspicious. A tag with lift near 1
is context, not evidence. Two lessons. First, the tag order written by intuition (EV gift first) was wrong; the
data says "folded the better hand" is the strongest single sign of a planted transfer. Second, the isolation
results predict section 8: when the defining action also happens innocently all the time, ranking the planted
isolation hands is hard.

➡️ **Consequence:** `TAG_ORDER` in `explain.py` now follows these measured lifts, and tags below ≈ 1.5 are dropped
for that family. The first tag an investigator reads is the most diagnostic one. `big_pot` / `big_flow` only fill
empty slots.

## 5 · The break-even cut: counting mistakes honestly

🎯 **Question:** how many mistakes does the model make, and of what kind?

🤔 A ranking has no mistakes until you pick a cut-off. We use the **break-even cut**: flag the K highest-risk pairs,
where K is the number of colluders (372). At this cut precision equals recall, and every missed colluder is matched
by exactly one flagged non-colluder, so false positives and false negatives can be compared one for one. We use the
**window** view (2,000-hand windows), the closest stand-in for the evaluation period.

In [ ]:
counts = E.outcome_counts(flagged)
print(json.dumps(counts, indent=1))
print(E.recall_by_family(flagged))

📊 **What we saw:** precision = recall ≈ 0.75 (280 of 372). **All 92 false positives are unlabeled pairs. Not one
of the 1,488 confirmed-innocent labeled pairs makes the top 372**; the best-ranked one sits at 385.

💡 **Inference:** the confirmed negatives look like random pairs (notebook 03), so this alone is weak evidence.
The striking part is where the false positives come from. The organizers say the labels "do not list every
coordinated pair", and P4's gap analysis estimated dozens of hidden colluders among exactly these unlabeled
high-risk pairs. Some of our "false positives" are probably true positives that the labels don't record.

➡️ **Consequence:** the local precision of 0.75 is a **lower bound**. This is also why the Kaggle score (≈ 0.84)
sits well above the local window final (≈ 0.65).

## 6 · False positives and the decoys: do innocent stories explain them?

🎯 **Question:** the data contains decoys: tilting players, weak or passive players, similar styles, pairs who
simply play together a lot. Are our false positives decoys that fooled the model?

🔬 **Method:** apply the benign-alternative templates (`BENIGN` in `explain.py`) to every pair. A template fires when
its trigger feature lies in the most extreme 10% of all pairs, so in a random pair each fires about 10% of the time.
Compare firing rates for true positives (TP), false positives (FP), missed colluders (FN) and true negatives (TN).

In [ ]:
rates, tops = E.decoy_profile(flagged)
print(rates)
print(tops.filter(pl.col("outcome") == "FP"))
for k, b in BENIGN.items():
    print(f"{k:<18} {b.title}  [{b.feature}, {b.tail} tail]")

📊 **What we saw** (rates per group; TN ≈ 10% for every template by construction):
- `passive_player` fires on **0%** of false positives (vs 11% of TNs) and only 6% of true positives.
- `neighbours` (17%) and `schedule_overlap` (13%) are mildly over-represented among false positives.
- For **37% of false positives no template fits at all**: no innocent story from our list explains them.

💡 **Inference:**
1. **The lift principle works on the classic decoy.** Passive players fold to everyone, which is the raw signature
   of directed transfer. The model never flags them, because it measures folds *against the partner* relative to
   folds against everyone else (notebook 02).
2. **Exposure decoys leak a little.** Pairs seated side by side, or together very often, have more interactions
   and so more chances for a lift to look large by chance. That is the decoy this model is most exposed to.
3. **The "no story fits" group** are the best candidates for hidden colluders (section 5).

➡️ **Consequence:** in the investigator view, `neighbours` and `schedule_overlap` deserve the most attention as
benign alternatives. A future feature idea: shrink lifts more strongly for adjacent seats.

## 7 · False negatives: what the missed colluders have in common

🎯 **Question:** 92 colluders were missed. Are they a random 25%, or do they share something?

🔬 **Method:** for each family, compare the median of its defining features (and of exposure) between caught (TP)
and missed (FN) colluders, all in the 2,000-hand window view.

In [ ]:
mp = E.missed_profile(flagged)
print(mp)
print(rates.filter(pl.col("outcome").is_in(["TP", "FN"])).select("outcome", "small_sample", "schedule_overlap"))

📊 **What we saw:**
- **Directed transfer:** missed pairs moved far fewer chips (median net flow ≈ 94 vs 395 bb/100) and gave up much
  less expected value against the partner (`ev_gift_lift_max` ≈ 0.8 vs 3.2). They also shared fewer hands.
- **Soft play:** the median missed pair has `hu_strong_check_lift` = **0**. They never had a strong hand heads-up
  against each other inside the window, so the defining signal could not occur.
- **Isolation:** missed pairs have weaker presence and squeeze lifts (`vpip_pres_max` 0.07 vs 0.10).
- `small_sample` fires on 17% of missed colluders vs 7% of caught ones.

💡 **Inference:** most misses are not the model ignoring a visible signal. They are pairs whose **coordination was
light or had few chances to show**: small transfers, too few hands, no heads-up spot with a strong hand. A ranking
model cannot rank a signal that isn't in the data.

➡️ **Consequence:** two different fixes. (1) More sensitive *hand-level* signals help the light colluders: a few
very suspicious hands can stand out even when averages don't (that was P4's hand-suspicion idea). (2) For pairs
with few chances, no feature will help; the honest response is to show the `small_sample` warning.

## 8 · Behavior and evidence mistakes by family

In [ ]:
true, predicted, m = E.behavior_confusion(oof)
print("window view, labeled colluders (rows = true family):")
print(pl.DataFrame(m, schema=predicted, orient="row").insert_column(0, pl.Series("true", true)))
print(E.evidence_by_family())

📊 **What we saw:** the behavior confusion is almost diagonal: once a pair is known to collude, M2 names the family
correctly about 98% of the time. Evidence is a different story. Evidence AP@5 is ≈ 0.53 for directed transfer,
≈ 0.60 for soft play and only ≈ 0.18 for isolation, **even though 88% of isolation pairs have at least one planted
hand in their top 5**.

💡 **Inference:** Behavior MAP is limited by the *risk ranking*, not by the family call. Isolation evidence is weak
because planted isolation hands look like the pair's other hands: one opens, the other folds behind, which also
happens innocently all the time. M4 finds *a* planted hand but cannot put the planted ones in the right order.

➡️ **Consequence:** the largest remaining headroom is still isolation evidence, which matches P4's gap analysis.

## 9 · What goes on the public model card

`python -m suspoker.export_web --model-card` writes `web/public/data/model_card.json`. It contains **only
aggregates**: the results table, repeated-CV means, Kaggle scores, PR curves (101 points each), the confusion
matrix, global importances, the error counts and profiles from this notebook, and the tag and template
definitions. Before writing, it searches its own text for anything shaped like a competition ID (hand, pair,
player or table) and refuses to write if it finds one (tested in `tests/test_explain.py`).

In [ ]:
from suspoker.export_web import WEB_DATA

card = json.loads((WEB_DATA / "model_card.json").read_text(encoding="utf-8"))
print(list(card))
print(card["errors"]["counts"])

## 10 · Summary

| # | Finding | Why it matters |
|---|---|---|
| 1 | SHAP contributions add up exactly to each raw score | Explanations are the model itself, not an approximation |
| 2 | Controls (style) rank high in importance, mostly as *downward* pushes | Importance ≠ suspicion; show reasons against too |
| 3 | 0 of 1,488 confirmed negatives in the top 372; all 92 FPs are unlabeled | Local precision 0.75 is a lower bound (hidden colluders) |
| 4 | Passive players never become FPs; neighbours / high overlap slightly over-represented | The lift principle beats the classic decoy; exposure is the weak spot |
| 5 | Missed colluders are light or had few chances (no heads-up strong hand, small flows) | Most misses are signals absent from the data, not ignored |
| 6 | Behavior ≈ 98% right given a colluder; isolation evidence AP ≈ 0.18 | Remaining headroom: risk ranking and isolation evidence |

### Self-check
1. Why is additivity the property that makes a per-pair explanation trustworthy?
2. A feature has the highest mean |SHAP|. Why doesn't that mean high values of it are suspicious?
3. Why does the break-even cut make false positives and false negatives directly comparable?
4. Why are "false positives" among unlabeled pairs not all mistakes, and how does that change the local precision?
5. Why do passive players, who fold to everyone, almost never get flagged for directed transfer?
6. A soft-play pair never had a strong hand heads-up against each other. What can the model do about it, and what
   should the investigator be told?